# Lesson 5

## further development of makemore model

In [2]:
import sys
from pathlib import Path

# Add the parent directory (nnz2h) to sys.path
parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))

import pygrad

import math
import numpy as np
import matplotlib.pyplot as plt
import random
import torch # import torch for the usage of their tensors
import statistics
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# jupyter magic:
%matplotlib inline

Multi-layer perception model (introduces a hyper-parameter)

- changes the neurons by that amount that is fully connected and is editable and is optimized for loss function

- rough layout is the table -> first neuron layer -> softmax (turn to probability) which is all backpropagated

In [3]:
words = open('names.txt','r').read().splitlines()

In [4]:
# buidl the vocab and mapping to and from integers

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [5]:
# build dataset

# supports the prediction (3 inputs for the 4th)

block_size = 3 # context length: how many chars do we take to predict the next one?
X, Y = [], []

for w in words[:5]:
    print(w)
    context = [0] * block_size # 0 tokens
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context) # stores running context
        Y.append(ix)
        print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [6]:
X.shape, X.dtype, Y.shape, Y.dtype # X is indvidiual examples, Y is the label

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

Create a neural network that takes this X to predict the Y

In [7]:
# 27 possible characters (embed them in a lower dimensional space)

C = torch.randn((27,2)) # embedded matrix, lookup

# example of indexing into the C matrix

F.one_hot(torch.tensor(5),num_classes=27).float() @ C # think of first layer of neural net or integer indexing into table C

emb = C[X] # for every 32x3 integers, an embedding vector was retrieved for it



In [8]:
# hidden layer

W1 = torch.randn((6,100)) # 2 dim embedding and 3 incoming neurons, there first is 6, second is up to us

b1 = torch.randn(100) # biases

# emb @ W1 + b1 wont work since 32,3,2 wont multiply by W1

# must transform emb to 32x6

emb[:, 0, :] # plucks out 32 by 2, ignores 3

torch.cat([emb[:, 0 , :], emb[:, 1 , :], emb[:, 2 , :]],1) # does not generalize

torch.cat((torch.unbind(emb, 1)), 1) # another way to do it that is generalized

emb.view(32,6) # most efficient way to concatenate due to it not storing anything in memory

# -1 makes torch infer what it should be
h = torch.tanh(emb.view(-1,6) @ W1 + b1)  # activation matrix

In [9]:
# second layer
W2 = torch.randn(100,27) # 100 determined by h, 27 by character amount
b2 = torch.randn(27)

logits = h @ W2 + b2

counts = logits.exp()
prob = counts/counts.sum(1,keepdim=True)

In [10]:
loss = - prob[torch.arange(32),Y].log().mean() # gives current probabilities to the correcct character

# boom the loss was developed

In [12]:
# to make it replicatable lets make it have a set seed

g = torch.Generator().manual_seed(2147863647)
C = torch.randn((27,2),generator=g)
W1 = torch.randn((6,100),generator=g)
b1 = torch.randn(100,generator=g)
W2 = torch.randn((100,27),generator=g)
b2 = torch.randn(27,generator=g)
parameters = [C,W1,b1,W2,b2]

In [13]:
for p in parameters:
    p.requires_grad = True

In [ ]:
# basic loop with 5 words

for _ in range(10):

    # forward pass

    emb = C[X] # (32, 3 ,2)
    h = torch.tanh(emb.view(-1,6) @ W1 + b1)
    logits = h @ W2 + b2 # (32, 27)

    # old

    # counts = logits.exp()
    # prob = counts / counts.sum(1, keepdims=True)
    # loss = -prob[torch.arrange(32), Y].log().mean()

    # new

    loss = F.cross_entropy(logits,Y)

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    #update
    for p in parameters:
        p.data += -0.1 * p.grad

Lets now train it on a full model

In [ ]:
# Establish database

X = []
Y = []

for w in words:
    context = [0] * block_size # 0 tokens
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context) # stores running context
        Y.append(ix)
        context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)




torch.Size([228146, 3])

In [20]:
for _ in range(100):

    # forward pass

    emb = C[X] # (32, 3 ,2)
    h = torch.tanh(emb.view(-1,6) @ W1 + b1)
    logits = h @ W2 + b2 # (32, 27)

    loss = F.cross_entropy(logits,Y)

    print(loss.item())

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    #update
    for p in parameters:
        p.data += -0.1 * p.grad

10.444462776184082
9.628101348876953
9.165472984313965
8.814239501953125
8.496174812316895
8.20150089263916
7.927062034606934
7.670899391174316
7.431382179260254
7.20713996887207
6.997152805328369
6.800670623779297
6.6170735359191895
6.445705413818359
6.285818576812744
6.136584281921387
5.997157573699951
5.866722583770752
5.744532585144043
5.629917144775391
5.522283554077148
5.42110538482666
5.325899124145508
5.236220359802246
5.1516499519348145
5.071792125701904
4.996279716491699
4.924776554107666
4.856982707977295
4.792631149291992
4.731485366821289
4.67333984375
4.6180100440979
4.565329551696777
4.515145778656006
4.467313289642334
4.421694278717041
4.378153324127197
4.336557865142822
4.296778678894043
4.2586870193481445
4.222159385681152
4.187076091766357
4.153322696685791
4.120793342590332
4.089388847351074
4.059020042419434
4.029604911804199
4.001071453094482
3.9733564853668213
3.9464035034179688
3.9201648235321045
3.894599676132202
3.8696703910827637
3.8453481197357178
3.82160592

Notice it is a bit slow with data, to avoid this issue we split the data into small batches and iterate through